# cGAN — 128×128 (Model A, No MediaPipe) — OPTIMIZED

**Optimized notebook — same math, faster execution.** Every change below is a
*speed* optimization only; the architecture, loss functions and training
schedule are identical to the original notebook, so results are equivalent.

### Optimizations applied (see the 🚀 notes through the notebook)
1. **Mixed precision (float16 + loss scaling)** — uses the GPU's tensor cores; output/logit/softmax kept in float32.
2. **`tf.data` pipeline + prefetch** — overlaps host→device copies with GPU compute (no per-step blocking `tf.constant`).
3. **In-graph one-hot + vectorized target selection** — removes the per-step Python `np.random.choice` loop.
4. **Optional XLA** (`USE_XLA`) — fuses kernels for extra speed.

> Toggle any optimization off in the **Hyperparameters** cell to reproduce the
> original float32 path exactly.

## Section 1 — Environment

In [ ]:
!pip install tensorflow==2.19.0 opencv-python --quiet
!pip install pytorch-fid scikit-image scikit-learn tqdm lpips --quiet
!pip install torch torchvision --quiet
print('Installed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, csv, warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import Counter
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from skimage.metrics import structural_similarity as ssim_fn

warnings.filterwarnings('ignore', category=UserWarning)
print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

## Section 2 — Hyperparameters & speed toggles

In [ ]:
# ── Speed toggles (flip OFF to reproduce the original float32 path) ──────────
USE_MIXED_PRECISION = True    # float16 compute + loss scaling
USE_XLA             = False   # set True for extra speed if stable on your GPU
TF_DATA_PREFETCH    = True

# ── Architecture ────────────────────────────────────────────────────────────
Z_DIM, IMG_SIZE, IMG_CHANNELS = 128, 128, 1

# ── Training ────────────────────────────────────────────────────────────────
EPOCHS, BATCH_SIZE = 50, 32
LR_G, LR_D = 2e-4, 1e-4
LR_DECAY_G, LR_DECAY_D = 35, 20
LABEL_SMOOTH = 0.9
G_UPDATES_BASE, G_D_RATIO_MAX = 2, 2.0

# Pixel structural loss schedule
LAMBDA_PIX_START, LAMBDA_PIX_END = 0.5, 5.0
WARMUP_EP, PHASE2_EP = 10, 10

# Evaluation
N_FID_PER_CLASS, N_FID_SEEDS, N_PKLE_PER_CLASS, SAVE_EVERY_N = 60, 5, 15, 5

### 🚀 Speed setup (mixed precision) — run before building models

In [ ]:
# 🚀 OPTIMIZATION 1 — speed setup. Call BEFORE building any model so the
# mixed-precision policy applies to every layer that gets created afterwards.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try: tf.config.experimental.set_memory_growth(_g, True)
    except RuntimeError: pass

if USE_MIXED_PRECISION and _gpus:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision: mixed_float16 ENABLED (tensor cores)')
else:
    print('Mixed precision: OFF (float32)')
print('XLA jit_compile:', 'ON' if USE_XLA else 'OFF')

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/cgan_A_128_opt'
DATA_PATH  = '/content/drive/MyDrive/ArASL_Database_54K_Final/ArASL_Database_54K_Final'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE,'checkpoints')
SAMPLES_DIR = os.path.join(DRIVE_BASE,'samples')
EVAL_DIR = os.path.join(DRIVE_BASE,'eval')
REAL_FID_DIR = os.path.join(EVAL_DIR,'fid_real')
HISTORY_DIR = os.path.join(DRIVE_BASE,'history')
PLOTS_DIR = os.path.join(DRIVE_BASE,'plots')
PROGRESS_FILE = os.path.join(CHECKPOINT_DIR,'progress.json')
for d in [CHECKPOINT_DIR,SAMPLES_DIR,EVAL_DIR,REAL_FID_DIR,HISTORY_DIR,PLOTS_DIR]: os.makedirs(d, exist_ok=True)
print('Dirs ready.')

## Section 3 — Data loading

In [ ]:
VALID_EXT = ('.png', '.jpg', '.jpeg')
imgs, labs, errs = [], [], 0
print('Loading from', DATA_PATH)
for sub in tqdm(sorted(os.listdir(DATA_PATH)), desc='Classes'):
    spath = os.path.join(DATA_PATH, sub)
    if not os.path.isdir(spath): continue
    for fn in sorted(os.listdir(spath)):
        if not fn.lower().endswith(VALID_EXT): continue
        try:
            raw = tf.io.read_file(os.path.join(spath, fn))
            img = tf.image.decode_png(raw, channels=1)
            img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
            img = (tf.cast(img, tf.float32) - 127.5) / 127.5
            imgs.append(img.numpy()); labs.append(sub)
        except Exception:
            errs += 1
images_array_pre = np.asarray(imgs, dtype=np.float32)
labels_array_pre = np.asarray(labs)
print('Loaded', len(images_array_pre), 'images,', errs, 'errors')

In [ ]:
label_encoder = LabelEncoder()
labels_int_pre = label_encoder.fit_transform(labels_array_pre)
num_classes = len(label_encoder.classes_)
label_to_idx = {l: i for i, l in enumerate(label_encoder.classes_)}
idx_to_label = {i: l for l, i in label_to_idx.items()}

# deterministic shuffle (one-time)
rng = np.random.default_rng(RANDOM_SEED)
perm = rng.permutation(len(images_array_pre))
images_array = images_array_pre[perm]
labels_int   = labels_int_pre[perm]
print('Classes:', num_classes, '|', list(label_encoder.classes_))
total = len(images_array)
for cls in sorted(Counter(labels_array_pre)):
    n = (labels_array_pre == cls).sum()
    print(f'  {cls:10s}: {n:5d} ({n/total*100:.1f}%)')

In [ ]:
class_image_prototypes = np.zeros((num_classes, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
class_real_indices = {}
for c in range(num_classes):
    idx = np.where(labels_int == c)[0]
    class_real_indices[c] = idx
    if len(idx): class_image_prototypes[c] = images_array[idx].mean(axis=0)
class_img_proto_tf = tf.constant(class_image_prototypes)
print('Prototypes built:', class_image_prototypes.shape)

## Section 4 — 🚀 Optimized input pipeline (tf.data)

In [ ]:
# 🚀 OPTIMIZATION 2 + 3 — tf.data pipeline with prefetch, in-graph one-hot,
# and a vectorized per-epoch structural-target selector.
#
# WHY FASTER: the original built `tf.constant(slice)` + `tf.one_hot(...)` on the
# host every step and blocked on the copy; and it chose the Phase-2 target with a
# per-element Python np.random.choice loop. Here gather + one-hot happen inside a
# prefetched pipeline, and the whole epoch's targets are chosen in one vectorized
# shot. Same images, same batches, same selection rule -> identical training data.
AUTOTUNE = tf.data.AUTOTUNE
N = len(images_array)

class TargetSelector:
    """Vectorized Phase-1 (prototype) / Phase-2 (random real member) indexing.
    Targets index into source = concat([images, prototypes]) of length N + C:
      prototype of class c -> N + c ;  a real member of class c -> its own index."""
    def __init__(self, labels_int, num_classes, n_images):
        self.labels, self.N = labels_int, n_images
        order = np.argsort(labels_int, kind='stable')
        self.flat_members = order.astype(np.int64)
        self.counts = np.bincount(labels_int, minlength=num_classes)
        self.offsets = np.concatenate([[0], np.cumsum(self.counts)[:-1]]).astype(np.int64)
    def epoch_targets(self, epoch, phase2_ep, rng):
        if epoch < phase2_ep:
            return (self.N + self.labels).astype(np.int32)         # prototypes
        rnd = (rng.random(self.N) * self.counts[self.labels]).astype(np.int64)
        return self.flat_members[self.offsets[self.labels] + rnd].astype(np.int32)

# single host constant reused as BOTH input source and target source
_source = np.concatenate([images_array, class_image_prototypes], axis=0)
source_tf = tf.constant(_source)
labels_tf = tf.constant(labels_int, dtype=tf.int32)
target_idx_var = tf.Variable(np.zeros(N, np.int32), trainable=False)

def _map_fn(idx):
    img  = tf.gather(source_tf, idx)
    oneh = tf.one_hot(tf.gather(labels_tf, idx), num_classes)   # in-graph one-hot
    tgt  = tf.gather(source_tf, tf.gather(target_idx_var, idx))
    return img, oneh, tgt

ds = (tf.data.Dataset.from_tensor_slices(tf.range(N, dtype=tf.int32))
        .shuffle(N, seed=RANDOM_SEED, reshuffle_each_iteration=True)
        .batch(BATCH_SIZE, drop_remainder=True)
        .map(_map_fn, num_parallel_calls=AUTOTUNE))
if TF_DATA_PREFETCH:
    ds = ds.prefetch(AUTOTUNE)

selector = TargetSelector(labels_int, num_classes, N)
del _source
print('tf.data pipeline ready:', ds.element_spec)

## Section 5 — Model architecture (mixed-precision safe)

In [ ]:
class SelfAttention2D(layers.Layer):
    """SAGAN self-attention. gamma=0 init. Softmax in float32 for fp16 safety."""
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        ch8 = max(channels // 8, 1)
        self.q = layers.Conv2D(ch8, 1, use_bias=False)
        self.k = layers.Conv2D(ch8, 1, use_bias=False)
        self.v = layers.Conv2D(channels, 1, use_bias=False)
        self.gamma = None
    def build(self, input_shape):
        self.gamma = self.add_weight(name='gamma', shape=(), initializer='zeros', trainable=True)
        super().build(input_shape)
    def call(self, x):
        B = tf.shape(x)[0]; H, W, C = x.shape[1], x.shape[2], x.shape[3]
        ch8 = max(C // 8, 1)
        q = tf.reshape(self.q(x), [B, H*W, ch8])
        k = tf.reshape(self.k(x), [B, H*W, ch8])
        v = tf.reshape(self.v(x), [B, H*W, C])
        scale = tf.cast(ch8, tf.float32) ** -0.5
        # 🚀 OPTIMIZATION 1 detail: softmax computed in float32 even under fp16
        logits = tf.cast(tf.matmul(q, k, transpose_b=True), tf.float32) * scale
        attn = tf.cast(tf.nn.softmax(logits, axis=-1), v.dtype)
        out = tf.reshape(tf.matmul(attn, v), [B, H, W, C])
        return x + self.gamma * out
    def get_config(self):
        cfg = super().get_config(); cfg['channels'] = self.channels; return cfg

In [ ]:
def build_generator(z_dim, num_classes):
    noise_in = tf.keras.Input(shape=(z_dim,), name='noise')
    label_in = tf.keras.Input(shape=(num_classes,), name='label')
    lbl = layers.Dense(128, use_bias=False)(label_in); lbl = layers.LeakyReLU(0.2)(lbl)
    x = layers.Concatenate()([noise_in, lbl])
    x = layers.Dense(4*4*512, use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Reshape((4,4,512))(x)
    x = layers.Conv2DTranspose(256,4,2,padding='same',use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(128,4,2,padding='same',use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(128,4,2,padding='same',use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = SelfAttention2D(128, name='self_attn')(x)
    x = layers.Conv2DTranspose(64,4,2,padding='same',use_bias=False)(x);  x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(32,4,2,padding='same',use_bias=False)(x);  x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # 🚀 OPTIMIZATION 1 detail: keep the output head in float32 for a stable tanh
    out = layers.Conv2D(1,3,padding='same',dtype='float32')(x)
    out = layers.Activation('tanh', dtype='float32', name='img_out')(out)
    return tf.keras.Model([noise_in, label_in], out, name='Generator_128')

In [ ]:
def build_discriminator(num_classes):
    SN = layers.SpectralNormalization
    image_in = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name='image')
    label_in = tf.keras.Input(shape=(num_classes,), name='label')
    lp = layers.Dense(IMG_SIZE*IMG_SIZE)(label_in); lp = layers.Reshape((IMG_SIZE, IMG_SIZE, 1))(lp)
    x = layers.Concatenate()([image_in, lp])
    x = SN(layers.Conv2D(64,4,strides=2,padding='same'))(x); x = layers.LeakyReLU(0.2)(x)
    x = SN(layers.Conv2D(128,4,strides=2,padding='same'))(x); x = layers.LeakyReLU(0.2)(x); x = layers.Dropout(0.2)(x)
    x = SN(layers.Conv2D(256,4,strides=2,padding='same'))(x); x = layers.LeakyReLU(0.2)(x); x = layers.Dropout(0.2)(x)
    x = SN(layers.Conv2D(512,4,strides=2,padding='same'))(x); x = layers.LeakyReLU(0.2)(x); x = layers.Dropout(0.3)(x)
    x = SN(layers.Conv2D(512,4,strides=2,padding='same'))(x); x = layers.LeakyReLU(0.2)(x); x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    out = layers.Dense(1, dtype='float32')(x)   # float32 logits
    return tf.keras.Model([image_in, label_in], out, name='Discriminator_128')

In [ ]:
generator = build_generator(Z_DIM, num_classes)
discriminator = build_discriminator(num_classes)
print('G params', f'{generator.count_params():,}', '| D params', f'{discriminator.count_params():,}')

## Section 6 — Loss schedule

In [ ]:
def get_lambda(epoch):
    if epoch >= WARMUP_EP: return float(LAMBDA_PIX_END)
    t = epoch / max(WARMUP_EP, 1)
    return float(LAMBDA_PIX_START + t*(LAMBDA_PIX_END-LAMBDA_PIX_START))
for e in [0,5,10,20,50]: print(f'  epoch {e:3d}: lambda={get_lambda(e):.2f}')

In [ ]:
# 🚀 OPTIMIZATION 1 detail — optimizer + gradient helpers that transparently
# handle loss scaling. With USE_MIXED_PRECISION=False these are plain Adam +
# plain gradients (identical to the original numerical path).
def make_optimizer(lr):
    opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999, clipnorm=1.0)
    if USE_MIXED_PRECISION:
        opt = tf.keras.mixed_precision.LossScaleOptimizer(opt)
    return opt

def apply_loss(opt, loss, tape, variables):
    if USE_MIXED_PRECISION:
        scaled = opt.get_scaled_loss(loss)
        grads = opt.get_unscaled_gradients(tape.gradient(scaled, variables))
    else:
        grads = tape.gradient(loss, variables)
    opt.apply_gradients(zip(grads, variables))

def set_lr(opt, lr):
    (opt.inner_optimizer if USE_MIXED_PRECISION else opt).learning_rate.assign(lr)

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

## Section 7 — Visualization

In [ ]:
TEST_LABELS = [l for l in ['bb','ain','gaaf','al','ha'] if l in label_to_idx]
tf.random.set_seed(RANDOM_SEED)
EVAL_NOISE = tf.random.normal([max(len(TEST_LABELS),1), Z_DIM], seed=RANDOM_SEED)

def generate_and_save_images(model, epoch, save_dir):
    n = len(TEST_LABELS)
    if n == 0: return
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for i, lbl in enumerate(TEST_LABELS):
        loh = tf.one_hot([label_to_idx[lbl]], depth=num_classes)
        gen = model([EVAL_NOISE[i:i+1], loh], training=False)
        img = (gen[0,:,:,0].numpy()*127.5+127.5).clip(0,255).astype('uint8')
        axes[i].imshow(img, cmap='gray', vmin=0, vmax=255)
        axes[i].set_title(lbl, fontweight='bold'); axes[i].axis('off')
    plt.suptitle(f'Epoch {epoch}', y=1.02); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'epoch_{epoch:04d}.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

## Section 8 — 🚀 Optimized training loop

**What changed vs original:** train steps are `@tf.function(jit_compile=USE_XLA)` with mixed-precision loss scaling; data arrives from the prefetched `tf.data` pipeline; the structural target is selected vectorized per epoch via `target_idx_var`. The **loss math is unchanged**: `G = g_adv + lambda * mean(|fake - target|)`.

In [ ]:
def train_A(generator, discriminator):
    g_opt, d_opt = make_optimizer(LR_G), make_optimizer(LR_D)
    lam_v = tf.Variable(get_lambda(0), dtype=tf.float32, trainable=False)

    ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator, g_opt=g_opt, d_opt=d_opt)
    ckpt_mgr = tf.train.CheckpointManager(ckpt, CHECKPOINT_DIR, max_to_keep=5)
    start_ep = 0
    hist = {k: [] for k in ['d','g_adv','g_pixel','g_total','gd_ratio','lam']}
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: prog = json.load(f)
        start_ep = prog.get('last_epoch', 0)
        for k in hist: hist[k] = prog.get(k, [])
        if ckpt_mgr.latest_checkpoint: ckpt.restore(ckpt_mgr.latest_checkpoint); print('Resumed from', start_ep)

    @tf.function(jit_compile=USE_XLA)
    def step_d(real, lbs):
        noise = tf.random.normal([tf.shape(real)[0], Z_DIM])
        with tf.GradientTape() as t:
            fake = generator([noise, lbs], training=True)
            r = discriminator([real, lbs], training=True); f = discriminator([fake, lbs], training=True)
            d_loss = bce(tf.ones_like(r)*LABEL_SMOOTH, r) + bce(tf.zeros_like(f), f)
        apply_loss(d_opt, d_loss, t, discriminator.trainable_variables)
        return d_loss

    @tf.function(jit_compile=USE_XLA)
    def step_g(real, lbs, tgt):
        noise = tf.random.normal([tf.shape(real)[0], Z_DIM])
        with tf.GradientTape() as t:
            fake = generator([noise, lbs], training=True)
            f = discriminator([fake, lbs], training=True)
            g_adv = bce(tf.ones_like(f), f)
            g_pix = tf.reduce_mean(tf.abs(fake - tgt))
            g_loss = g_adv + lam_v * g_pix
        apply_loss(g_opt, g_loss, t, generator.trainable_variables)
        return g_adv, g_pix

    g_n = G_UPDATES_BASE
    for epoch in range(start_ep, EPOCHS):
        if epoch == LR_DECAY_D: set_lr(d_opt, LR_D/2)
        if epoch == LR_DECAY_G: set_lr(g_opt, LR_G/2)
        lam = get_lambda(epoch); lam_v.assign(lam)
        rng = np.random.default_rng(RANDOM_SEED + epoch)
        target_idx_var.assign(selector.epoch_targets(epoch, PHASE2_EP, rng))   # 🚀 vectorized targets

        ed, ega, egp = [], [], []
        for real, lbs, tgt in ds:                                              # 🚀 prefetched pipeline
            ed.append(float(step_d(real, lbs)))
            ga = gp = 0.0
            for _ in range(g_n):
                a, p = step_g(real, lbs, tgt); ga += float(a); gp += float(p)
            ega.append(ga/g_n); egp.append(gp/g_n)

        dm, gam, gpm = float(np.mean(ed)), float(np.mean(ega)), float(np.mean(egp))
        gtm = gam + lam*gpm; gd = gtm/max(dm, 0.01)
        for k, v in zip(['d','g_adv','g_pixel','g_total','gd_ratio','lam'], [dm,gam,gpm,gtm,gd,lam]): hist[k].append(v)
        phase = 'P1' if epoch < PHASE2_EP else 'P2'
        print(f'Ep{epoch+1}/{EPOCHS} [{phase}] D={dm:.4f} Ga={gam:.4f} Gp={gpm:.4f} Gt={gtm:.4f} G/D={gd:.2f}x lam={lam:.2f} nG={g_n}')
        g_n = (min(G_UPDATES_BASE+2,5) if gd > G_D_RATIO_MAX*1.5 else G_UPDATES_BASE+1 if gd > G_D_RATIO_MAX else G_UPDATES_BASE)

        if (epoch+1) % SAVE_EVERY_N == 0 or epoch == 0: generate_and_save_images(generator, epoch+1, SAMPLES_DIR)
        ckpt_mgr.save()
        with open(PROGRESS_FILE, 'w') as f: json.dump({'last_epoch': epoch+1, **hist}, f)
        generator.save_weights(os.path.join(CHECKPOINT_DIR, 'generator.weights.h5'))
        discriminator.save_weights(os.path.join(CHECKPOINT_DIR, 'discriminator.weights.h5'))
    print('Training complete.')
    return hist

In [ ]:
hist = train_A(generator, discriminator)

## Section 9 — Loss curves

In [ ]:
def plot_losses(hist):
    if not hist.get('d'): print('No history.'); return
    ep = range(1, len(hist['d'])+1)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    ax[0].plot(ep, hist['d'], label='D'); ax[0].plot(ep, hist['g_total'], label='G total'); ax[0].axhline(0.693, ls='--'); ax[0].legend(); ax[0].set_title('D & G'); ax[0].grid(alpha=.3)
    ax[1].plot(ep, hist['g_adv'], label='G adv'); ax[1].plot(ep, hist['g_pixel'], label='G pixel'); ax[1].legend(); ax[1].set_title('G components'); ax[1].grid(alpha=.3)
    ax[2].plot(ep, hist['gd_ratio']); ax[2].axhline(1.0, ls='--'); ax[2].set_title('G/D ratio'); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR,'A_loss_curves.png'), dpi=150, bbox_inches='tight'); plt.show(); plt.close()
plot_losses(hist)

## Section 10 — Evaluation (FID, SSIM, LPIPS, Diversity)

In [ ]:
try:
    from pytorch_fid import fid_score; import torch
except ImportError:
    import subprocess; subprocess.run(['pip','install','pytorch-fid','--quiet'])
    from pytorch_fid import fid_score; import torch
FID_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def save_balanced_real(imgs, lbls, out_dir, n_per=N_FID_PER_CLASS):
    os.makedirs(out_dir, exist_ok=True); cnt = 0
    for ci in range(num_classes):
        for i in np.where(lbls == ci)[0][:n_per]:
            im = (imgs[i,:,:,0]*127.5+127.5).clip(0,255).astype(np.uint8)
            Image.fromarray(np.stack([im,im,im],-1)).save(os.path.join(out_dir, f'r_{cnt:05d}.png')); cnt += 1

def compute_fid(gen, real_dir, eval_dir, n_runs=N_FID_SEEDS, n_per=N_FID_PER_CLASS):
    scores = []
    for seed in range(n_runs):
        gdir = os.path.join(eval_dir, f'fid_fake_s{seed:02d}'); os.makedirs(gdir, exist_ok=True)
        tf.random.set_seed(seed); cnt = 0
        for ci in range(num_classes):
            loh = tf.one_hot([ci]*n_per, depth=num_classes)
            noise = tf.random.normal([n_per, Z_DIM], seed=seed*100+ci)
            fakes = gen([noise, loh], training=False).numpy()
            for j in range(n_per):
                im = (fakes[j,:,:,0]*127.5+127.5).clip(0,255).astype(np.uint8)
                Image.fromarray(np.stack([im,im,im],-1)).save(os.path.join(gdir, f'f_{cnt:05d}.png')); cnt += 1
        fv = fid_score.calculate_fid_given_paths([real_dir, gdir], batch_size=32, device=FID_DEVICE, dims=2048)
        scores.append(fv); print(f'  Seed {seed}: FID={fv:.4f}')
    m, s = float(np.mean(scores)), float(np.std(scores))
    print(f'FID: {m:.4f} +/- {s:.4f}'); return m, s, scores

save_balanced_real(images_array, labels_int, REAL_FID_DIR)
fid_m, fid_s, fid_scores = compute_fid(generator, REAL_FID_DIR, EVAL_DIR)

In [ ]:
def compute_ssim(gen, n_per=N_PKLE_PER_CLASS, seed=RANDOM_SEED):
    tf.random.set_seed(seed); all_s, per = [], {}
    for cname, ci in sorted(label_to_idx.items()):
        ridx = np.where(labels_int == ci)[0][:n_per]
        if not len(ridx): continue
        loh = tf.one_hot([ci]*len(ridx), depth=num_classes)
        noise = tf.random.normal([len(ridx), Z_DIM], seed=seed+ci)
        fakes = gen([noise, loh], training=False).numpy()
        cls = []
        for i in range(len(ridx)):
            r = (images_array[ridx[i],:,:,0]+1)/2; f = (fakes[i,:,:,0]+1)/2
            cls.append(float(ssim_fn(r, f, data_range=1.0))); all_s.append(cls[-1])
        per[cname] = float(np.mean(cls))
    m, s = float(np.nanmean(all_s)), float(np.nanstd(all_s)); print(f'SSIM: {m:.4f} +/- {s:.4f}'); return m, s, per
ssim_m, ssim_s, ssim_per = compute_ssim(generator)

In [ ]:
LPIPS_OK = False
try:
    import lpips as lpips_lib; lpips_fn = lpips_lib.LPIPS(net='vgg'); LPIPS_OK = True
except ImportError:
    print('pip install lpips first')

def compute_lpips(gen, n_per=10, seed=RANDOM_SEED):
    if not LPIPS_OK: return float('nan'), float('nan'), {}
    import torch; tf.random.set_seed(seed); all_l, per = [], {}
    for cname, ci in sorted(label_to_idx.items()):
        ridx = np.where(labels_int == ci)[0][:n_per]
        if not len(ridx): continue
        loh = tf.one_hot([ci]*len(ridx), depth=num_classes)
        noise = tf.random.normal([len(ridx), Z_DIM], seed=seed+ci)
        fakes = gen([noise, loh], training=False).numpy(); cls = []
        for i in range(len(ridx)):
            prep = lambda a: torch.tensor(np.stack([a[:,:,0]]*3,0)[None], dtype=torch.float32)
            with torch.no_grad():
                d = lpips_fn(prep(images_array[ridx[i]]), prep(fakes[i])).item()
            cls.append(d); all_l.append(d)
        per[cname] = float(np.mean(cls))
    m, s = float(np.nanmean(all_l)), float(np.nanstd(all_l)); print(f'LPIPS: {m:.4f} +/- {s:.4f}'); return m, s, per
lpips_m, lpips_s, lpips_per = compute_lpips(generator)

In [ ]:
def compute_diversity(gen, n_per=10, seed=RANDOM_SEED):
    tf.random.set_seed(seed); all_d = []
    for cname, ci in sorted(label_to_idx.items()):
        loh = tf.one_hot([ci]*n_per, depth=num_classes)
        noise = tf.random.normal([n_per, Z_DIM], seed=seed+ci)
        fakes = gen([noise, loh], training=False).numpy().reshape(n_per, -1)
        dists = [float(np.mean(np.abs(fakes[i]-fakes[j]))) for i in range(n_per) for j in range(i+1, n_per)]
        all_d.append(float(np.mean(dists)) if dists else 0.0)
    m = float(np.mean(all_d)); print(f'Diversity: {m:.4f}'); return m
div_m = compute_diversity(generator)

## Section 11 — Save results

In [ ]:
def _nn(o):
    if isinstance(o, float) and o != o: return None
    if isinstance(o, dict): return {k:_nn(v) for k,v in o.items()}
    if isinstance(o, list): return [_nn(i) for i in o]
    return o
results = {'model':'cGAN-A 128 (No MP) OPTIMIZED',
          'fid':{'mean':fid_m,'std':fid_s}, 'ssim':{'mean':ssim_m,'std':ssim_s},
          'lpips':{'mean':lpips_m,'std':lpips_s}, 'diversity':{'mean':div_m},
          'speed':{'mixed_precision':USE_MIXED_PRECISION,'xla':USE_XLA}}
with open(os.path.join(HISTORY_DIR,'A_results.json'),'w') as f: json.dump(_nn(results), f, indent=2)
print('Saved results.'); print(results)